In [1]:
# Run this only the first time
import sys
!{sys.executable} -m pip install fastapi uvicorn[standard] motor pymongo \
    "pydantic[email]" "python-jose[cryptography]" \
    "passlib[bcrypt]" python-dotenv nest_asyncio requests

In [3]:
import os

# Change this path to where you put the backend files
os.chdir(r"C:\Users\DELL\Documents\smartcampus\backend")

# Verify the files are there
print("Files found:", os.listdir("."))

Files found: ['.idea', '.ipynb_checkpoints', 'auth_utils.py', 'database.py', 'dependencies.py', 'env', 'main.py', 'requirements.txt', 'routes', 'schemas.py', 'Untitled.ipynb', 'venv']


In [7]:
import secrets
import os

secret_key = secrets.token_hex(32)

line1 = "MONGO_URL=mongodb://localhost:27017"
line2 = "DB_NAME=smartcampus"
line3 = "SECRET_KEY=" + secret_key

env_content = line1 + "\n" + line2 + "\n" + line3

with open(".env", "w") as f:
    f.write(env_content)

print("✓ .env created successfully!")
print()
print("Contents:")
with open(".env") as f:
    print(f.read())

✓ .env created successfully!

Contents:
MONGO_URL=mongodb://localhost:27017
DB_NAME=smartcampus
SECRET_KEY=55cc4009e51e57a024c991c3700e85c3183e5cdebd844f606b4c8a78594ea38c


In [9]:
# Python needs this file to treat routes/ as a package
os.makedirs("routes", exist_ok=True)

if not os.path.exists("routes/__init__.py"):
    with open("routes/__init__.py", "w") as f:
        f.write("")

print("✓ routes/__init__.py ready")
print("✓ routes/ contains:", os.listdir("routes"))

✓ routes/__init__.py ready
✓ routes/ contains: ['announcements.py', 'auth.py', 'events.py', 'timetable.py', '__init__.py']


In [1]:
import pymongo

try:
    client = pymongo.MongoClient("mongodb://localhost:27017", serverSelectionTimeoutMS=3000)
    client.server_info()
    print("✓ MongoDB is running — connected!")
    client.close()
except Exception as e:
    print("✗ MongoDB NOT running!")
    print("  → Start it: open a terminal and run:  mongod")
    print(f"  Error: {e}")

✓ MongoDB is running — connected!


In [ ]:
import uvicorn
import nest_asyncio

nest_asyncio.apply()  # required for Jupyter

print("Starting SmartCampus API on port 8003...")
print("Docs available at: http://localhost:8003/docs")
print("Press the STOP button (■) in Jupyter to shut down")

uvicorn.run(
    "main:app",
    host="0.0.0.0",
    port=8003,
    reload=False,
    log_level="info"
)

Starting SmartCampus API on port 8003...
Docs available at: http://localhost:8003/docs
Press the STOP button (■) in Jupyter to shut down


INFO:     Started server process [19492]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8003 (Press CTRL+C to quit)


[DB] Connected to MongoDB at mongodb://localhost:27017
INFO:     127.0.0.1:59351 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:59351 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:62318 - "GET /api/announcements/ HTTP/1.1" 401 Unauthorized
INFO:     127.0.0.1:62318 - "GET /api/timetable/ HTTP/1.1" 401 Unauthorized
INFO:     127.0.0.1:56317 - "GET /api/announcements/ HTTP/1.1" 403 Forbidden
INFO:     127.0.0.1:56318 - "GET /api/timetable/ HTTP/1.1" 403 Forbidden
INFO:     127.0.0.1:51252 - "POST /api/auth/register HTTP/1.1" 201 Created
INFO:     127.0.0.1:51252 - "GET /api/announcements/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:51252 - "GET /api/timetable/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:54706 - "GET /api/announcements/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:54706 - "GET /api/timetable/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:54706 - "GET /api/announcements/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:54706 - "GET /api/events/ HTTP/1.1" 200 OK
INFO:     127.0.0.1:54706 - "GET /api/a

In [ ]:
# Run this in a NEW cell while Cell 6 is still running
import requests

BASE = "http://localhost:8003/api"

# 1. Register
res = requests.post(f"{BASE}/auth/register", json={
    "full_name": "Test User",
    "email": "test@univ.dz",
    "password": "password123"
})
print("Register:", res.status_code, res.json())

# 2. Login
res = requests.post(f"{BASE}/auth/login", json={
    "email": "test@univ.dz",
    "password": "password123"
})
data = res.json()
token = data["access_token"]
print("Login ✓ Token:", token[:30], "...")

# 3. Get announcements with token
headers = {"Authorization": f"Bearer {token}"}
res = requests.get(f"{BASE}/announcements/", headers=headers)
print("Announcements:", len(res.json()), "items")